In [28]:
import pandas as pd
import anthropic
import json
import time
import os
from dotenv import load_dotenv
import time
from IPython.display import HTML
from collections import Counter

# Load sample csv

In [ ]:
"""Omdat de sample van 100 met annotaties niet in de eerdere sample van 1000 zit pakken we de ene 100 en 900 van de ander."""

df_sample_gold = pd.read_csv("LLM/gold_standard_sample.csv.gz", compression="gzip")

df_annotated = pd.read_csv("label_studio/gold_standard_25.csv")
annotated_eclis = df_annotated['ecli'].tolist()

df_annotated_samples = df_sample_gold[df_sample_gold['ecli'].isin(annotated_eclis)]

# Annotatie Anthropic Haiku


In [18]:
USER_PROMPT_TEMPLATE = "Annotate all legal references in the following Dutch court judgment:\n\n{text}"

SYSTEM_PROMPT = """You are an expert annotator of Dutch legal texts. Your task is to identify all legal references in a court judgment, following the principle of the MINIMAL UNIQUE IDENTIFIER: only include in the span what is needed to uniquely identify the law or decision.

LABELS — exactly four types:

BWB — Dutch legislation (laws, articles, paragraphs, subsections). Treaties stored in the BWB (EVRM, IVBPR, EU Charter, Refugee Convention) also count as BWB.
- art. 6:162 BW
- artikel 138 lid 1 onderdeel b Gw
- art. 17-23 Mw (range — ONE span)
- art. 6:162-164 BW (multiple — ONE span)
- Wetboek van Strafrecht (full name)
- Awb, Sv, Rv, BW, Sr, Gw, Mw, Wro, Wvggz, EVRM (abbreviations alone — valid BWB)
- Less common abbreviations: Wnb, Aw, Wabo, ZW, WW, AWR, Vw, WAO, WWB, WIA, Bpb, DHW
- hoofdstuk 3 Wro, titel 1 afd. 2 Boek 3 BW
- paragraaf C1/2.9 van de Vreemdelingencirculaire 2000
- art. 3.72 lid 1 onderdeel a onder 2° Wet IB 2001
- Wet van 20 december 2002 (Belgian-style law with date)

ECLI — References to court decisions. ALSO includes HvJ EU case numbers and EHRM application numbers.
- ECLI:NL:HR:2023:847 (ECLI alone)
- LJN ZC0986, LJN BC2153, LJN AB7291 (older decisions — 7-character codes)
- C-5/08, T-123/19 (EU case numbers — annotate WITH instance + date: "HvJ EU 16 juli 2009, zaak C-5/08")
- 31365/96 (EHRM application — annotate WITH instance + date: "EHRM 5 oktober 2000, nr. 31365/96")
- NJ 1998/625 (journal citation when no ECLI/LJN exists)
- arrest Kühne-Heitz (case name — include "arrest" when adjacent)
- Endstra-arrest, Stokke/H3 Products-arrest (case-name-arrest compounds)

CELEX — European legislation only (NOT case law).
- Verordening (EU) nr. 1215/2012
- Richtlijn 2011/95/EU
- CELEX:32012R1215
- art. 7 Verordening (EU) nr. 1215/2012 (article in regulation — ONE span)
- Brussel I bis, Rome I, AVG (short names)
- Informal Dutch names like "Auteursrechtrichtlijn 2001/29" → CELEX

OEP — Official publications.
- Stb. 2023, 506
- Staatsblad 2023, 506
- Stcrt. 2020, 12345
- Kamerstukken II 1997/98, 25 900, nr. 6 (full identifier — all parts needed)
- Kamerstukken I 2020/21, 35 570, nr. A
- Handelingen II 2020/21, nr. 45, item 3

==================================================================
CRITICAL RULE 1 — ECLI/LJN MINIMALISATION (most often violated!)
==================================================================

When you see an ECLI or LJN anywhere in a reference, annotate ONLY the identifier itself. Court + date + case name + journal all OUTSIDE the span.

Basic case:
"HR 12 maart 2020, ECLI:NL:HR:2020:847, NJ 2020/123"
  ❌ WRONG: "HR 12 maart 2020, ECLI:NL:HR:2020:847"
  ✅ CORRECT: "ECLI:NL:HR:2020:847"

LJN works the same way:
"HR 4 juni 1993, LJN ZC0986, NJ 1993/659"
  ❌ WRONG: "HR 4 juni 1993, LJN ZC0986"
  ✅ CORRECT: "LJN ZC0986"

"HR 30 mei 2008, LJN BC2153"
  ❌ WRONG: "HR 30 mei 2008, LJN BC2153"
  ✅ CORRECT: "LJN BC2153"

The minimalisation rule ALSO applies in these cases:

1. With case name (roepnaam) before the ECLI:
   "Endstra-arrest van 30 mei 2008, ECLI:NL:HR:2008:BC2153"
   ✅ Span: "ECLI:NL:HR:2008:BC2153"
   "Stokke/H3 Products-arrest van 22 februari 2013, ECLI:NL:HR:2013:BY1529"
   ✅ Span: "ECLI:NL:HR:2013:BY1529"

2. With lower courts (Rb., Hof, Gerechtshof, BenGH, CRvB, CBb):
   "Rb. Breda 16 mei 2012, ECLI:NL:RBBRE:2012:BW5808"
   ✅ Span: "ECLI:NL:RBBRE:2012:BW5808"
   "Gerechtshof Den Haag 19 juli 2016, ECLI:NL:GHDHA:2016:2137"
   ✅ Span: "ECLI:NL:GHDHA:2016:2137"

3. With case name in parentheses:
   "HR 12 april 2013 (Hauck/Stokke), ECLI:NL:HR:2012:BY1533"
   ✅ Span: "ECLI:NL:HR:2012:BY1533"
   The parenthetical case name is NEVER part of the span.

4. With LJN and EU/EHRM case number together — TWO spans:
   "HvJEU 16 juli 2009, nr. C-5/08, LJN BJ3749"
   ✅ TWO spans: "HvJEU 16 juli 2009, nr. C-5/08" AND "LJN BJ3749"

The ONLY exceptions (instance + date IN the span):
- EU case numbers: "HvJ EU 16 juli 2009, zaak C-5/08"
- EHRM application numbers: "EHRM 5 oktober 2000, nr. 31365/96"
- EHRM with appl. no.: "EHRM 8 oktober 2015, appl. no. 77212/12"

==================================================================
CRITICAL RULE 2 — CASE NAMES WITHOUT ECLI
==================================================================

When a case name (roepnaam) appears WITHOUT an ECLI/LJN nearby, annotate the case name itself:
"het Stokke/H3 Products-arrest" → span: "Stokke/H3 Products-arrest"
"het Endstra-arrest" → span: "Endstra-arrest"
"het arrest Kühne-Heitz" → span: "arrest Kühne-Heitz"

Rules:
- The article "het" is EXCLUDED from the span
- The suffix "-arrest" IS included when directly attached
- The word "arrest" IS included when it precedes the name with a space

==================================================================
CRITICAL RULE 3 — LJN RECOGNITION
==================================================================

LJN codes are 7-character identifiers (2 letters + 5 alphanumeric) used in older Dutch case law BEFORE ECLI existed. ALWAYS annotate them as ECLI label:
- "LJN ZC0986" → ECLI
- "LJN AB7291" → ECLI
- "LJN BC2153" → ECLI
- "LJN AX3171" → ECLI

==================================================================
CRITICAL RULE 4 — DEFINING ABBREVIATIONS IN PARENTHESES
==================================================================

When a full law name is followed by its abbreviation in parentheses, split into TWO separate spans (parentheses excluded):

"Algemene wet bestuursrecht (Awb)" 
  ❌ WRONG: one span "Algemene wet bestuursrecht (Awb)"
  ✅ CORRECT: TWO spans → "Algemene wet bestuursrecht" + "Awb"

"Wet natuurbescherming (Wnb)" → "Wet natuurbescherming" + "Wnb"
"Wet verplichte geestelijke gezondheidszorg (Wvggz)" → "Wet verplichte..." + "Wvggz"
"Europees Verdrag... (EVRM)" → "Europees Verdrag..." + "EVRM"

Same rule applies to article references with defining abbreviation:
"artikel 8:55d, tweede lid, van de Algemene wet bestuursrecht (Awb)"
  ❌ WRONG: one span including "(Awb)"
  ✅ CORRECT: TWO spans → "artikel 8:55d, tweede lid, van de Algemene wet bestuursrecht" + "Awb"

==================================================================
CRITICAL RULE 5 — LAW NAMES ALWAYS RECOGNISED
==================================================================

These law names should ALWAYS be annotated whenever they appear, even when not explicitly defined in the text:
- Auteurswet (also: "Aw")
- Wet natuurbescherming (also: "Wnb")
- Wet Bopz (Wet bijzondere opnemingen in psychiatrische ziekenhuizen)
- Wet verplichte geestelijke gezondheidszorg (also: "Wvggz")
- Participatiewet
- WVW 1994 (Wegenverkeerswet 1994)
- DHW (Drank- en Horecawet)
- Wet Bibob

Same for any reference that includes these names with article numbers:
- "art. 326 Sr"
- "art. 5 EVRM" / "art. 6 lid 1 EVRM"
- "art. 10 Aw" / "Art. 1 van de Auteurswet"

==================================================================

SPAN BOUNDARIES — MINIMAL UNIQUE IDENTIFIER:

Outside the span (NEVER include):
- Leading articles/prepositions: "het", "de", "in", "zie", "van"
- Punctuation at start/end: commas, parentheses, periods
- Version indicators in parentheses: "(oud)", "(nieuw)", "(thans)", "(zoals dat luidde tot...)"
- Descriptive context: "als bedoeld in...", "waarbij vastgesteld dat...", "de in [...] bedoelde eisen"
- Title descriptions after CELEX numbers (e.g. "betreffende de rechterlijke bevoegdheid...")
- Page numbers (p. 12) and type indicators ((MvT)) for OEP

For multiple references — same vs different laws:
- Same law shared → ONE span: "art. 6 en 7 EVRM" is ONE span
- Same law shared → ONE span: "artikelen 33, 33a en 420bis Sr" is ONE span
- Same law (juncto, same name) → ONE span: "artikel 222 lid 2 juncto 220 lid 2 juncto artikel 353 lid 1 Rv"
- Different laws → SEPARATE spans: "artikel 2 en 99 Rv juncto artikel 1:14 BW" → "artikel 2 en 99 Rv" + "artikel 1:14 BW"
- "artikel 220 Rv jo artikel 222 Rv" → ONE span (same law)

For anaphoric references (back-references):
- Annotate "artikel 1:251a lid 1 onder a" WITHOUT law name IF the law was named earlier in the same context.
- Annotate "artikel 2 van de wet" when "de wet" was earlier established.
- Do NOT annotate pure anaphoric references without article numbers: "die wet", "dat artikel", "dit wetsartikel" alone.

DO NOT ANNOTATE:
- References in the "inhoudsindicatie" (summary section) — only the judgment text itself
- Internal references to the judgment: "zie overweging 3.2", "zoals hierboven overwogen"
- General legal terms without specific basis: "de wet", "het verdrag", "het gelijkheidsbeginsel", "het vertrouwensbeginsel", "redelijkheid en billijkheid"
  EXCEPTION: do annotate when followed by specific basis: "het gelijkheidsbeginsel van artikel 1 Grondwet" → annotate "artikel 1 Grondwet"
- Party names, lawyer names, judge names
- Dutch internal case numbers: "SGR AWB 19/3196", "C/13/123456" (these are process numbers, NOT decision identifiers)
- Pure anaphoric references ("die wet", "dat artikel") alone
- Court decisions identified ONLY by instance + date with no ECLI/LJN/journal/roepnaam: "het vonnis van 3 juni 2021 van de rechtbank Overijssel" → do NOT annotate
- Within cited statutory text in appendices: do NOT annotate article headings ("Artikel 38" as subheading) or internal references ("van de wet", "dit artikel")

DECISION RULES for edge cases:
- BWB vs OEP overlap → use most specific class. Law itself = BWB. Publication in Staatsblad = OEP.
- EU rule transposed into Dutch law → European text intended = CELEX. Dutch implementation = BWB.
- CELEX vs ECLI for EU law → legislation = CELEX. Court decisions (HvJ EU, EHRM) = ECLI.
- Foreign law (Belgium, Curaçao, etc.) → foreign statute = BWB, foreign decision = ECLI.
- Malformed identifiers (typos, missing colons, double country codes) → annotate as written: "ECLI:NLGHAMS:2017:2163", "ECLI:NL:HR: 2017:1187".

GENERAL PRINCIPLES:
- Be recall-oriented: when in doubt, include the span.
- Copy spans EXACTLY as they appear — character for character, including spacing and punctuation.
- Annotate EVERY occurrence of a reference, even if the same span appears multiple times.

OUTPUT FORMAT — return ONLY a JSON array. Each item has exactly two fields:
- "span": the exact text copied from the judgment
- "label": one of BWB, ECLI, CELEX, OEP

Example output:
[
  {"span": "artikel 8:55d, tweede lid, van de Algemene wet bestuursrecht", "label": "BWB"},
  {"span": "Awb", "label": "BWB"},
  {"span": "Wet natuurbescherming", "label": "BWB"},
  {"span": "Wnb", "label": "BWB"},
  {"span": "Auteurswet", "label": "BWB"},
  {"span": "art. 326 Sr", "label": "BWB"},
  {"span": "ECLI:NL:HR:2023:847", "label": "ECLI"},
  {"span": "LJN ZC0986", "label": "ECLI"},
  {"span": "Stokke/H3 Products-arrest", "label": "ECLI"},
  {"span": "HvJ EU 16 juli 2009, zaak C-5/08", "label": "ECLI"},
  {"span": "EHRM 5 oktober 2000, nr. 31365/96", "label": "ECLI"},
  {"span": "Richtlijn 2011/95/EU", "label": "CELEX"},
  {"span": "Stb. 2023, 506", "label": "OEP"},
  {"span": "Kamerstukken II 1997/98, 25 900, nr. 6", "label": "OEP"}
]

If no legal references are found, return an empty array: []"""

In [19]:
from dotenv import load_dotenv
import os
import json
import time
import anthropic

load_dotenv()
claude_key = os.getenv("claude_key")
os.environ["ANTHROPIC_API_KEY"] = claude_key
client = anthropic.Anthropic()

# ── Hulpfuncties ──────────────────────────────────────────────────────────────

def zoek_offsets(tekst, annotations, ecli=None, gemiste_log=None):
    """Zoekt offsets van spans in de tekst. Logt gemiste spans als gemiste_log gegeven is."""
    resultaat = []
    zoekpositie = 0

    for ann in annotations:
        if not isinstance(ann, dict) or "span" not in ann or "label" not in ann:
            continue

        span = ann["span"]

        pos = tekst.find(span, zoekpositie)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            zoekpositie = pos + 1
            continue

        pos = tekst.find(span)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            continue

        print(f"  ⚠ [{ecli}] Span niet gevonden: '{span}'")
        if gemiste_log is not None:
            gemiste_log.append({
                "ecli": ecli,
                "span": span,
                "label": ann["label"]
            })

    return resultaat


def chunk_tekst(tekst, max_chars=24000):
    """Splits tekst in stukken op alinea-grenzen."""
    if len(tekst) <= max_chars:
        return [tekst]

    paragraphs = tekst.split("\n\n")
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) > max_chars:
            if current:
                chunks.append(current.strip())
            current = para
        else:
            current += "\n\n" + para
    if current:
        chunks.append(current.strip())
    return chunks


def annotate_judgment(tekst, ecli=None, gemiste_log=None):
    """Annoteert een uitspraak via Claude Haiku."""
    chunks = chunk_tekst(tekst)
    all_annotations = []
    offset = 0

    for chunk in chunks:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=8192,
            system=SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(text=chunk)
            }]
        )

        raw = response.content[0].text.strip()

        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                annotations = parsed
            elif isinstance(parsed, dict):
                annotations = next((v for v in parsed.values() if isinstance(v, list)), [])
            else:
                annotations = []

            annotations_met_offsets = zoek_offsets(chunk, annotations, ecli=ecli, gemiste_log=gemiste_log)
            for ann in annotations_met_offsets:
                ann["start"] += offset
                ann["end"] += offset
            all_annotations.extend(annotations_met_offsets)

        except json.JSONDecodeError as e:
            print(f"  Parse error op offset {offset}: {e}")

        offset += len(chunk) + 2
        time.sleep(0.3)

    return all_annotations


# ── Hoofdloop ─────────────────────────────────────────────────────────────────

OUTPUT_FILE = "LLM/gold_standard_haiku_25.json"
GEMISTE_FILE = "LLM/gemiste_spans_haiku_25.json"

# Laad al geannoteerde ECLIs zodat je kunt hervatten na onderbreking
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        results = json.load(f)
    al_gedaan = {r["ecli"] for r in results}
    print(f"Hervat: {len(al_gedaan)} uitspraken al gedaan")
else:
    results = []
    al_gedaan = set()

# Laad bestaande gemiste spans als die er zijn
if os.path.exists(GEMISTE_FILE):
    with open(GEMISTE_FILE) as f:
        gemiste_spans = json.load(f)
else:
    gemiste_spans = []

failed = []

for i, row in df_annotated_samples.iterrows():
    if row["ecli"] in al_gedaan:
        continue

    try:
        annotations = annotate_judgment(
            row["tekst"],
            ecli=row["ecli"],
            gemiste_log=gemiste_spans
        )
        results.append({
            "ecli": row["ecli"],
            "annotations": annotations,
            "n_annotations": len(annotations)
        })
        al_gedaan.add(row["ecli"])

        if len(al_gedaan) % 10 == 0:
            print(f"{len(al_gedaan)}/{len(df_annotated_samples)} — laatste: {len(annotations)} annotaties")

        # Sla tussentijds op na elke 10 uitspraken
        if len(al_gedaan) % 10 == 0:
            with open(OUTPUT_FILE, "w") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            with open(GEMISTE_FILE, "w") as f:
                json.dump(gemiste_spans, f, ensure_ascii=False, indent=2)

        time.sleep(0.5)

    except Exception as e:
        print(f"Fout bij {row['ecli']}: {e}")
        failed.append(row["ecli"])

# Sla definitief op
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(GEMISTE_FILE, "w") as f:
    json.dump(gemiste_spans, f, ensure_ascii=False, indent=2)

print(f"\nKlaar. {len(results)} uitspraken geannoteerd, {len(failed)} mislukt.")
print(f"Totaal gemiste spans: {len(gemiste_spans)}")
if failed:
    print(f"Mislukt: {failed}")

  ⚠ [ECLI:NL:RBDHA:2020:808] Span niet gevonden: 'Awb'
  ⚠ [ECLI:NL:RBDHA:2022:13136] Span niet gevonden: 'artikel 8:55d, tweede lid, van de Algemene wet bestuursrecht'
  ⚠ [ECLI:NL:RBDHA:2022:13136] Span niet gevonden: 'Awb'
  ⚠ [ECLI:NL:RBDHA:2022:13136] Span niet gevonden: 'artikel 8.11 NBW'
  ⚠ [ECLI:NL:RBDHA:2022:13136] Span niet gevonden: 'Artikel 8 Verordening (EU) nr. 1215/2012'
  ⚠ [ECLI:NL:RBDHA:2022:13136] Span niet gevonden: 'artikel VI.83 en VI.84 van het Wetboek van Economisch Recht'
10/25 — laatste: 16 annotaties
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 9'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 14a'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 14b'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 14c'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 22b'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 22c'
  ⚠ [ECLI:NL:RBROT:2018:934] Span niet gevonden: 'artikel 22d'
  ⚠ [ECLI:NL:RBROT:2018:934

# Annotaties checken

In [11]:
def toon_annotaties(ecli, results, df_sample):
    result = next(r for r in results if r["ecli"] == ecli)
    tekst = df_sample[df_sample["ecli"] == ecli]["tekst"].iloc[0]
    
    kleuren = {
        "BWB":   "#fff388f4",
        "ECLI":  "#90caf9",
        "CELEX": "#a5d6a7",
        "OEP":   "#ffcc80"
    }
    
    anns = sorted(result["annotations"], key=lambda x: (x["start"], -x["end"]))
    
    html = "<style>mark { padding: 2px; border-radius: 3px; }</style>"
    html += "<p style='font-family: Arial; line-height: 1.8;'>"
    
    positie = 0
    for ann in anns:
        if ann["start"] < positie:
            continue
        html += tekst[positie:ann["start"]]
        
        kleur = kleuren.get(ann["label"], "#e0e0e0")
        
        html += (f'<mark style="background:{kleur}; color:black" '
                 f'title="{ann["label"]}">'
                 f'{tekst[ann["start"]:ann["end"]]}</mark>')
        positie = ann["end"]
    
    html += tekst[positie:] + "</p>"
    
    # Legenda
    legenda_items = []
    for label, kleur in kleuren.items():
        legenda_items.append(f'<span style="background:{kleur}; color:black; padding:3px 6px; border-radius:3px">{label}</span>')
    legenda = " ".join(legenda_items)
    
    return HTML(f"<b>{ecli}</b> — {result['n_annotations']} annotaties<br>{legenda}<br><br>{html}")

In [12]:
for ecli in annotated_eclis[:3]:
    display(toon_annotaties(ecli, results, df_annotated_samples))

## Duplicates toevoegen

In [21]:
def verwijder_overlappen(annotations):
    """Bewaart bij overlappende spans altijd de langste."""
    if not annotations:
        return []
    
    # Sorteer op startpositie, langste span eerst bij gelijke start
    gesorteerd = sorted(annotations, key=lambda x: (x["start"], -(x["end"])))
    
    resultaat = []
    laatste_end = -1
    
    for ann in gesorteerd:
        if ann["start"] >= laatste_end:
            # Geen overlap — voeg toe
            resultaat.append(ann)
            laatste_end = ann["end"]
        else:
            # Overlap — vervang vorige als deze langer is
            if ann["end"] > laatste_end:
                resultaat[-1] = ann
                laatste_end = ann["end"]
    
    return resultaat

def voeg_duplicaten_toe(tekst, annotations):
    # Bewaar originele posities
    originele_starts = {a["start"] for a in annotations}
    
    lokale_gazetteer = {}
    for ann in annotations:
        if ann["span"] not in lokale_gazetteer:
            lokale_gazetteer[ann["span"]] = ann["label"]
    
    alle_annotations = []
    for span, label in lokale_gazetteer.items():
        pos = 0
        while True:
            pos = tekst.find(span, pos)
            if pos == -1:
                break
            alle_annotations.append({
                "span": span,
                "label": label,
                "start": pos,
                "end": pos + len(span),
                "is_duplicate": pos not in originele_starts  # Nieuw veld
            })
            pos += 1
    
    alle_annotations.sort(key=lambda x: x["start"])
    return verwijder_overlappen(alle_annotations)

In [24]:
for result in results:
    ecli = result["ecli"]
    tekst = df_annotated_samples[df_annotated_samples["ecli"] == ecli]["tekst"].iloc[0]
    
    uitgebreid = voeg_duplicaten_toe(tekst, result["annotations"])
    result["annotations"] = uitgebreid
    result["n_annotations"] = len(uitgebreid)

print(f"Klaar. Totaal annotaties: {sum(r['n_annotations'] for r in results)}")

# Sla op
with open("LLM/gold_standard_haiku_25_dedup.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

Klaar. Totaal annotaties: 313
